# Módulo 4 — Correlaciones: ONI y Precipitación vs IRAG
**Objetivo:** calcular la correlación de Spearman entre ONI/precipitación y casos de IRAG  
para cada municipio con lags 0–3, e identificar patrones espaciales.

Contenido:
1. Correlación nacional (serie agregada)
2. Correlación por municipio — ONI vs IRAG
3. Correlación por municipio — Precipitación vs IRAG
4. Top 10 municipios por correlación
5. Comparación ONI vs precipitación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# ── Configuración ─────────────────────────────────────────────────────────
DATASET_PATH = (
    r"C:\Users\laura\OneDrive\TESIS\ETL_LauraChacon"
    r"\ETL_code\dashboard_3\processed\join\dataset_irag_2018_2024.parquet"
)

MAX_LAG  = 3    # rezagos a evaluar (0, 1, 2, 3)
MIN_OBS  = 24   # mínimo de observaciones para calcular correlación
TOP_N    = 10   # municipios a mostrar en rankings

COLORES = {
    "irag":    "#6B3A9E",
    "precip":  "#185FA5",
    "oni":     "#E24B4A",
    "neutral": "#888780",
}

NOMBRES_MESES = ["Ene","Feb","Mar","Abr","May","Jun",
                 "Jul","Ago","Sep","Oct","Nov","Dic"]

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "font.size":        10,
})
print("Librerías cargadas")

In [ ]:
# ── Carga ─────────────────────────────────────────────────────────────────
df = pd.read_parquet(DATASET_PATH)
df["date"] = pd.to_datetime(df["date"])
df["mes"]  = df["date"].dt.month
df["anio"] = df["date"].dt.year
df = df.sort_values(["muni_code", "date"]).reset_index(drop=True)

print(f"Dataset cargado: {len(df):,} registros · "
      f"{df['muni_code'].nunique()} municipios · "
      f"{df['anio'].min()}–{df['anio'].max()}")

## 1. Correlación nacional (serie agregada)

In [ ]:
def correlacion_cruzada(serie_x: pd.Series, serie_y: pd.Series,
                        max_lag: int, min_obs: int) -> pd.DataFrame:
    """
    Correlación de Spearman entre serie_x desplazada y serie_y
    para lags 0 a max_lag. Retorna DataFrame con resultados por lag.
    """
    resultados = []
    for lag in range(0, max_lag + 1):
        x_lag = serie_x.shift(lag)
        mask  = x_lag.notna() & serie_y.notna()
        if mask.sum() < min_obs:
            continue
        r, p = spearmanr(x_lag[mask], serie_y[mask])
        resultados.append({
            "lag":           lag,
            "correlacion":   round(r, 4),
            "p_valor":       round(p, 4),
            "significativo": p < 0.05,
        })
    return pd.DataFrame(resultados)


# Agregar a nivel nacional
nacional = (
    df.groupby("date")
    .agg(
        casos        = ("casos",         "sum"),
        precip_media = ("precip_mean_mm", "mean"),
        oni          = ("value_oni",      "first"),
    )
    .reset_index()
    .sort_values("date")
)

# Correlaciones nacionales
corr_oni_nac    = correlacion_cruzada(nacional["oni"],          nacional["casos"], MAX_LAG, MIN_OBS)
corr_precip_nac = correlacion_cruzada(nacional["precip_media"], nacional["casos"], MAX_LAG, MIN_OBS)

print("CORRELACIÓN NACIONAL — ONI vs casos IRAG:")
print(corr_oni_nac.to_string(index=False))
print()
print("CORRELACIÓN NACIONAL — Precipitación vs casos IRAG:")
print(corr_precip_nac.to_string(index=False))

## 2. Correlación por municipio — ONI vs IRAG

In [ ]:
def correlacion_municipio(grupo: pd.DataFrame, var_x: str,
                          max_lag: int, min_obs: int) -> dict | None:
    """
    Para un municipio calcula Spearman entre var_x y casos
    para lags 0 a max_lag. Retorna el lag con mayor correlación absoluta.
    """
    grupo = grupo.sort_values("date")
    mejor = {"corr": 0.0, "lag": 0, "p_valor": 1.0}

    for lag in range(0, max_lag + 1):
        x    = grupo[var_x].shift(lag)
        y    = grupo["casos"]
        mask = x.notna() & y.notna()
        if mask.sum() < min_obs:
            continue
        r, p = spearmanr(x[mask], y[mask])
        if abs(r) > abs(mejor["corr"]):
            mejor = {"corr": r, "lag": lag, "p_valor": p}

    if mejor["corr"] == 0.0:
        return None
    return {
        "corr":    round(mejor["corr"],    4),
        "lag":     mejor["lag"],
        "p_valor": round(mejor["p_valor"], 4),
        "sig":     mejor["p_valor"] < 0.05,
    }


print("Calculando correlación ONI vs IRAG por municipio...")
print("(puede tardar 1–2 minutos)")

registros_oni = []
for muni, grupo in df.groupby("muni_code"):
    res = correlacion_municipio(grupo, "value_oni", MAX_LAG, MIN_OBS)
    if res:
        res["muni_code"] = muni
        registros_oni.append(res)

df_corr_oni = pd.DataFrame(registros_oni)
df_corr_oni = df_corr_oni[["muni_code","corr","lag","p_valor","sig"]]

print(f"\nMunicipios procesados         : {len(df_corr_oni):,}")
print(f"Con correlación positiva sig. : {((df_corr_oni['corr'] > 0) & df_corr_oni['sig']).sum()}")
print(f"Con correlación negativa sig. : {((df_corr_oni['corr'] < 0) & df_corr_oni['sig']).sum()}")
print(f"Sin significancia             : {(~df_corr_oni['sig']).sum()}")
print(f"\nDistribución de correlaciones:")
print(df_corr_oni["corr"].describe().round(4))

## 3. Correlación por municipio — Precipitación vs IRAG

In [ ]:
print("Calculando correlación precipitación vs IRAG por municipio...")

registros_precip = []
for muni, grupo in df.groupby("muni_code"):
    res = correlacion_municipio(grupo, "precip_mean_mm", MAX_LAG, MIN_OBS)
    if res:
        res["muni_code"] = muni
        registros_precip.append(res)

df_corr_precip = pd.DataFrame(registros_precip)
df_corr_precip = df_corr_precip[["muni_code","corr","lag","p_valor","sig"]]

print(f"\nMunicipios procesados         : {len(df_corr_precip):,}")
print(f"Con correlación positiva sig. : {((df_corr_precip['corr'] > 0) & df_corr_precip['sig']).sum()}")
print(f"Con correlación negativa sig. : {((df_corr_precip['corr'] < 0) & df_corr_precip['sig']).sum()}")
print(f"Sin significancia             : {(~df_corr_precip['sig']).sum()}")
print(f"\nDistribución de correlaciones:")
print(df_corr_precip["corr"].describe().round(4))

## 4. Top 10 municipios por correlación

In [ ]:
def imprimir_top(df_corr: pd.DataFrame, variable: str,
                 n: int = 10) -> None:
    """Imprime top N municipios con mayor correlación positiva y negativa."""
    sig = df_corr[df_corr["sig"]]

    top_pos = sig.nlargest(n, "corr")[["muni_code","corr","lag","p_valor"]]
    top_neg = sig.nsmallest(n, "corr")[["muni_code","corr","lag","p_valor"]]

    print(f"TOP {n} correlación POSITIVA — {variable} → IRAG")
    print("=" * 55)
    print(f"{'#':<4} {'muni_code':<12} {'r':>8} {'lag':>5} {'p':>8}")
    print("-" * 55)
    for i, row in top_pos.reset_index(drop=True).iterrows():
        print(f"{i+1:<4} {row['muni_code']:<12} "
              f"{row['corr']:>8.4f} "
              f"{int(row['lag']):>5} "
              f"{row['p_valor']:>8.4f}")

    print(f"\nTOP {n} correlación NEGATIVA — {variable} → IRAG")
    print("=" * 55)
    print(f"{'#':<4} {'muni_code':<12} {'r':>8} {'lag':>5} {'p':>8}")
    print("-" * 55)
    for i, row in top_neg.reset_index(drop=True).iterrows():
        print(f"{i+1:<4} {row['muni_code']:<12} "
              f"{row['corr']:>8.4f} "
              f"{int(row['lag']):>5} "
              f"{row['p_valor']:>8.4f}")


imprimir_top(df_corr_oni,    "ONI",          TOP_N)
print()
imprimir_top(df_corr_precip, "Precipitación", TOP_N)

## 5. Comparación ONI vs precipitación

In [ ]:
# Unir las dos tablas para comparar
df_comp = df_corr_oni.merge(
    df_corr_precip,
    on="muni_code",
    suffixes=("_oni", "_precip"),
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Comparación correlaciones por municipio · IRAG Inusitada",
             fontsize=12, fontweight="bold")

# Histograma correlaciones ONI
axes[0].hist(
    df_corr_oni["corr"], bins=40,
    color=COLORES["oni"], alpha=0.7, edgecolor="white"
)
axes[0].axvline(0, color="gray", linewidth=1, linestyle="--")
axes[0].set_xlabel("Correlación Spearman")
axes[0].set_ylabel("Municipios")
axes[0].set_title("ONI → IRAG", fontweight="bold")
axes[0].grid(axis="y", alpha=0.3)

# Histograma correlaciones precipitación
axes[1].hist(
    df_corr_precip["corr"], bins=40,
    color=COLORES["precip"], alpha=0.7, edgecolor="white"
)
axes[1].axvline(0, color="gray", linewidth=1, linestyle="--")
axes[1].set_xlabel("Correlación Spearman")
axes[1].set_ylabel("Municipios")
axes[1].set_title("Precipitación → IRAG", fontweight="bold")
axes[1].grid(axis="y", alpha=0.3)

# Scatter ONI vs precipitación correlaciones
sig_ambos = df_comp[df_comp["sig_oni"] & df_comp["sig_precip"]]
sig_ninguno = df_comp[~df_comp["sig_oni"] & ~df_comp["sig_precip"]]
sig_solo_oni = df_comp[df_comp["sig_oni"] & ~df_comp["sig_precip"]]
sig_solo_precip = df_comp[~df_comp["sig_oni"] & df_comp["sig_precip"]]

axes[2].scatter(sig_ninguno["corr_oni"],    sig_ninguno["corr_precip"],
                color="lightgray", alpha=0.4, s=10, label="Ninguno sig.")
axes[2].scatter(sig_solo_oni["corr_oni"],   sig_solo_oni["corr_precip"],
                color=COLORES["oni"], alpha=0.6, s=15, label="Solo ONI sig.")
axes[2].scatter(sig_solo_precip["corr_oni"],sig_solo_precip["corr_precip"],
                color=COLORES["precip"], alpha=0.6, s=15, label="Solo precip. sig.")
axes[2].scatter(sig_ambos["corr_oni"],      sig_ambos["corr_precip"],
                color=COLORES["irag"], alpha=0.8, s=20, label="Ambos sig.")

axes[2].axhline(0, color="gray", linewidth=0.8, linestyle="--")
axes[2].axvline(0, color="gray", linewidth=0.8, linestyle="--")
axes[2].set_xlabel("Correlación ONI → IRAG")
axes[2].set_ylabel("Correlación Precipitación → IRAG")
axes[2].set_title("ONI vs Precipitación por municipio", fontweight="bold")
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nResumen de significancia:")
print(f"  Ambos significativos      : {len(sig_ambos):,}")
print(f"  Solo ONI significativo    : {len(sig_solo_oni):,}")
print(f"  Solo precip. significativo: {len(sig_solo_precip):,}")
print(f"  Ninguno significativo     : {len(sig_ninguno):,}")

In [ ]:
# Guardar resultados para usar en Módulo 5 (análisis espacial)
OUTPUT_ONI    = (
    r"C:\Users\laura\OneDrive\TESIS\ETL_LauraChacon"
    r"\ETL_code\dashboard_3\processed\join\corr_oni_irag.parquet"
)
OUTPUT_PRECIP = (
    r"C:\Users\laura\OneDrive\TESIS\ETL_LauraChacon"
    r"\ETL_code\dashboard_3\processed\join\corr_precip_irag.parquet"
)

df_corr_oni.to_parquet(OUTPUT_ONI,    index=False)
df_corr_precip.to_parquet(OUTPUT_PRECIP, index=False)

print(f"✓ Correlaciones ONI guardadas    : {OUTPUT_ONI}")
print(f"✓ Correlaciones precip. guardadas: {OUTPUT_PRECIP}")